# MMALS EcoSpec 1.0 Smoke Notebook

This notebook exercises the reference contract: context belief, adaptive complexity, goal-conditioned route energy, hard constraints, and reconstructive trace generation. It does **not** train the full MMALS backbone.

In [ ]:
# Run from the repository root.
# !pip -q install -e .[dev]
from mmals_ecospec import ContextBelief, EcoController, EcosystemState, GoalWeights, RouteCandidate
from mmals_ecospec.routing import RouteConstraints

In [ ]:
state = EcosystemState(
    step=1, latent=(0.2, -0.1, 0.8),
    context=ContextBelief({'C1': 0.48, 'C2': 0.44, 'C3': 0.08}),
    drift=0.24, risk=0.12,
)
routes = [
    RouteCandidate('precision', 0.97, 0.68, 0.82, 0.72, 0.55, risk=0.12),
    RouteCandidate('retention', 0.82, 0.97, 0.55, 0.91, 0.62, risk=0.08),
    RouteCandidate('compiled_specialist', 0.79, 0.81, 0.14, 0.80, 0.96, risk=0.10),
]
constraints = RouteConstraints(min_accuracy=0.78, max_risk=0.25)
controller = EcoController()

In [ ]:
goals = {
 'accuracy': GoalWeights(0.70, 0.10, 0.05, 0.10, 0.05),
 'retention': GoalWeights(0.10, 0.70, 0.05, 0.10, 0.05),
 'cost': GoalWeights(0.10, 0.05, 0.70, 0.10, 0.05),
 'specialization': GoalWeights(0.10, 0.10, 0.05, 0.10, 0.65),
}
for label, goal in goals.items():
    selected, trace = controller.decide(state, goal, routes, constraints)
    print(label, '->', selected.route_id if selected else 'ABSTAIN', '|', trace.complexity_mode.value)